# Laboratorio 11. Ejemplo integrado de soporte

Ejemplo final que combina contexto del sistema, rutas, JSON, logging, fechas, resumen de estados, hash y backup.

## 1. Preparar rutas del tema

In [ ]:
from pathlib import Path


def localizar_tema10() -> Path:
    """Localiza el directorio Tema10 aunque el notebook se ejecute desde notebooks, Tema10 o Curso_Python."""
    actual = Path.cwd().resolve()
    candidatos = [
        actual,
        actual.parent,
        actual / "Tema10",
        actual.parent / "Tema10",
    ]

    for candidato in candidatos:
        if candidato.name == "Tema10" and candidato.exists():
            return candidato
        if (candidato / "src").exists() and (candidato / "notebooks").exists():
            return candidato

    raise RuntimeError("No se ha podido localizar el directorio Tema10. Abre el notebook desde Tema10/notebooks o desde el proyecto Curso_Python.")


base_dir = localizar_tema10()
src_dir = base_dir / "src"
data_dir = base_dir / "data"
output_dir = base_dir / "output"
backup_dir = base_dir / "backup"
log_dir = base_dir / "logs"

for directorio in (data_dir, output_dir, backup_dir, log_dir):
    directorio.mkdir(parents=True, exist_ok=True)

print("base_dir:", base_dir)
print("data_dir:", data_dir)

## 2. Importar módulos estándar usados en el flujo integrado

In [ ]:
import os
import sys
import json
import shutil
import hashlib
import logging
import platform
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

## 3. Configurar directorios y logging

In [ ]:
for directorio in (data_dir, output_dir, backup_dir, log_dir):
    directorio.mkdir(parents=True, exist_ok=True)

log_file = log_dir / "operacion_integrada.log"

logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,
)
print("Log integrado:", log_file)

## 4. Definir funciones auxiliares

In [ ]:
def obtener_contexto():
    """Devuelve información básica del entorno de ejecución."""
    return {
        "python": sys.version.split()[0],
        "ejecutable": sys.executable,
        "sistema": platform.system(),
        "release": platform.release(),
        "maquina": platform.machine(),
        "usuario": os.environ.get("USER") or os.environ.get("USERNAME") or "desconocido",
        "directorio_actual": os.getcwd(),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }


def puerto_valido(puerto):
    """Valida un puerto TCP/UDP."""
    return 1 <= puerto <= 65535


def normalizar_servicios(servicios):
    """Normaliza registros de servicio y añade estado de validación."""
    normalizados = []

    for servicio in servicios:
        nombre = servicio["nombre"].strip().lower()
        puerto = servicio["puerto"]
        activo = servicio["activo"]

        if activo and puerto_valido(puerto):
            estado = "OK"
        elif not puerto_valido(puerto):
            estado = "PUERTO_INVALIDO"
        else:
            estado = "DETENIDO"

        normalizados.append({
            "nombre": nombre,
            "host": servicio["host"],
            "puerto": puerto,
            "activo": activo,
            "estado": estado,
        })

    return normalizados


def calcular_huella(ruta):
    """Calcula SHA-256 del fichero generado."""
    contenido = ruta.read_bytes()
    return hashlib.sha256(contenido).hexdigest()

## 5. Preparar datos de entrada simulados

In [ ]:
servicios = [
    {"nombre": " SSH ", "host": "srv-web-01", "puerto": 22, "activo": True},
    {"nombre": "NGINX", "host": "srv-web-01", "puerto": 443, "activo": True},
    {"nombre": "api", "host": "srv-app-01", "puerto": 8080, "activo": False},
    {"nombre": "backup", "host": "srv-bkp-01", "puerto": 70000, "activo": True},
]
servicios

## 6. Ejecutar procesamiento

In [ ]:
logging.info("Inicio de operación integrada.")

contexto = obtener_contexto()
registros = normalizar_servicios(servicios)
resumen = Counter(registro["estado"] for registro in registros)

salida = {
    "contexto": contexto,
    "total_servicios": len(registros),
    "resumen_estados": dict(resumen),
    "servicios": registros,
}

print("Resumen:", dict(resumen))
registros

## 7. Escribir JSON, calcular huella y generar backup

In [ ]:
ruta_json = output_dir / "reporte_soporte.json"
ruta_json.write_text(json.dumps(salida, indent=4, ensure_ascii=False), encoding="utf-8")

huella = calcular_huella(ruta_json)
copia = backup_dir / "reporte_soporte.json"
shutil.copy2(ruta_json, copia)

logging.info("Reporte generado: %s", ruta_json)
logging.info("Backup generado: %s", copia)
logging.info("SHA-256 del reporte: %s", huella)
logging.info("Fin de operación integrada.")

print("JSON     :", ruta_json)
print("Backup   :", copia)
print("Log      :", log_file)
print("SHA-256  :", huella)

## 8. Revisar salida estructurada

In [ ]:
contenido = json.loads(ruta_json.read_text(encoding="utf-8"))
print("Total servicios:", contenido["total_servicios"])
print("Estados:", contenido["resumen_estados"])
print("Primer servicio:", contenido["servicios"][0])